In [1]:
!pip install pandas openpyxl

import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


# Estructura

## Subsistemas

El sistema eléctrico español consta de sistemas eléctricos. De ellos, el subsistema eléctrico peninsular el de mayor tamaño y por el que pasa el mayor volumen de energía eléctrica: más de 200 GWh al año. En el caso de los territorios no peninsulares, debido a su pequeño tamaño y su carácter aislado, cuentan con características especiales que requieren de una operación singular. Nos referimos a los subsistemas de:

In [2]:
# https://www.esios.ree.es/es/descargas
file_path = 'Excels/133-Subsistemas.xlsx'

# Read the Excel file
subsistemas_df = pd.read_excel(file_path)

# Display the dataframe without the index
subsistemas_df.style.hide(axis='index')

Código,Código Numérico,Descripción
SB,81,Baleares
CE,70,Ceuta
FL,71,Fuerteventura-Lanzarote
GC,72,Gran Canaria
HI,73,Hierro
IF,74,Ibiza-Formentera
LG,75,La Gomera
PA,78,La Palma
MM,77,Mallorca-Menorca
ML,76,Melilla


In [3]:
print(f"Hay {len(subsistemas_df)} subsistemas no peninsulares")

Hay 12 subsistemas no peninsulares


## Enlaces nacionales

## Interconexiones internacionales

# Infraestructuras

## Subestaciones

Información de REE

In [4]:
# Split the 'Nombre y tensión del nudo' column into two parts, handling spaces in the name
def split_nudo(row):
    if pd.isna(row) or not isinstance(row, str):
        return pd.Series([None, None])
    # Split on the last space to handle names with spaces
    parts = row.rsplit(' ', 1)  # rsplit splits from the right, taking only the last space
    if len(parts) == 2:
        return pd.Series([parts[0], parts[1]])
    else:
        return pd.Series([row, None])  # If no split is possible, keep original as Nombre, None for Tensión

In [5]:
# https://www.ree.es/es/clientes/generador/acceso-conexion/conoce-la-capacidad-de-acceso
# Specify the path to your Excel file
file_path = 'Excels/Capacidad_de_acceso_a_RdT_ED_01ago25.xlsx'  # Adjust the path based on where your file is

# Read the Excel file
df_capacidad = pd.read_excel(file_path)

# Create a new dataframe with the first two columns and rows starting from the fourth Excel row (0-based index 2 onwards)
subestaciones_df = df_capacidad.iloc[2:, :2].copy()

# Reset the index for the new dataframe (optional, but cleans it up)
subestaciones_df.reset_index(drop=True, inplace=True)

subestaciones_df.to_excel('Excels/subestaciones.xlsx', index=False)

subestaciones_df

,Nombre y tensión del nudo,Comunidad Autónoma
0,ABADES 400,Castilla y León
1,ABADES 220,Castilla y León
2,ABADIANO 220,País Vasco
3,ABANILLAS 400,Región de Murcia
4,ABANTO 400,País Vasco
...,...,...
932,ZARZON 400,Extremadura
933,ZIERBENA 400,País Vasco
934,ZONA FRANCA 220,Cataluña
935,ZUMARRAGA 220,País Vasco


In [6]:
# Split the 'Nombre y tensión del nudo' column into two columns
subestaciones_df[['Nombre', 'Tensión del nudo']] = subestaciones_df['Nombre y tensión del nudo'].apply(split_nudo)

# Create a new DataFrame with all original columns plus the new split columns
subestaciones_df = subestaciones_df.drop(columns=['Nombre y tensión del nudo']).copy()

# Convert 'Tensión del nudo' to numeric type (optional, if you need it as numbers)
subestaciones_df['Tensión del nudo'] = pd.to_numeric(subestaciones_df['Tensión del nudo'], errors='coerce')

# Create a new DataFrame with 'Nombre' as the first column, 'Tensión del nudo' as the second, followed by other columns
subestaciones_tension_df = subestaciones_df[['Nombre', 'Tensión del nudo','Comunidad Autónoma']].copy()

subestaciones_tension_df.to_excel('Excels/subestaciones_tension.xlsx', index=False)

# Display the dataframe without the index
subestaciones_tension_df

,Nombre,Tensión del nudo,Comunidad Autónoma
0,ABADES,400,Castilla y León
1,ABADES,220,Castilla y León
2,ABADIANO,220,País Vasco
3,ABANILLAS,400,Región de Murcia
4,ABANTO,400,País Vasco
...,...,...,...
932,ZARZON,400,Extremadura
933,ZIERBENA,400,País Vasco
934,ZONA FRANCA,220,Cataluña
935,ZUMARRAGA,220,País Vasco


Nombres de las subestaciones, eliminando filas con el mismo nombre y distinta Tensión de nudo

In [10]:
# Group by Nombre and Comunidad Autónoma
# Concatenate values of 'Tensión del nudo' in the order they appear
nombre_subestaciones_df = (
    subestaciones_tension_df.groupby(["Nombre", "Comunidad Autónoma"], as_index=False)
      .agg({"Tensión del nudo": lambda x: "/".join(map(str, x))})
)

nombre_subestaciones_df 

,Nombre,Comunidad Autónoma,Tensión del nudo
0,ABADES,Castilla y León,400/220
1,ABADIANO,País Vasco,220
2,ABANILLAS,Región de Murcia,400
3,ABANTO,País Vasco,400
4,ABEGONDO,Galicia,400/220
...,...,...,...
801,ZARZON,Extremadura,400
802,ZIERBENA,País Vasco,400
803,ZONA FRANCA,Cataluña,220
804,ZUMARRAGA,País Vasco,220


Información de Zenodo

In [8]:
# https://zenodo.org/records/14144752
file_path = 'CSVs/buses.csv'

# Read the Excel file
df_buses_europa = pd.read_csv(file_path)

# Create a new dataframe with only rows where country="ES", keeping all columns
df_buses_ES = df_buses_europa[(df_buses_europa['country'] == 'ES') & (df_buses_europa['under_construction'] == 'f')].copy()

df_buses_ES.to_csv('CSVs/buses_ES.csv', index=False)

df_buses_ES

,bus_id,voltage,dc,symbol,under_construction,tags,x,y,country,geometry
487,ES1-400,400,f,Substation,f,ES1,-7.478964,43.702028,ES,POINT (-7.478963550093904 43.7020276396567)
488,ES10-220,220,f,Substation,f,ES10,-3.849633,43.344541,ES,POINT (-3.849633099999998 43.3445405893267)
489,ES100-220,220,f,Substation,f,ES100,1.344685,42.318896,ES,POINT (1.3446848999999976 42.31889588845269)
490,ES101-220,220,f,Substation,f,ES101,-8.138020,42.300963,ES,POINT (-8.138019599999994 42.30096278843702)
491,ES102-220,220,f,Substation,f,ES102,-8.080452,42.288093,ES,POINT (-8.080452199999987 42.28809258842574)
...,...,...,...,...,...,...,...,...,...,...
6698,way/986410945-400,400,f,Substation,f,way/986410945,-6.560160,40.612552,ES,POINT (-6.560160300536124 40.61255239371956)
6722,way/989699715-400,400,f,Substation,f,way/989699715,-6.360559,39.799277,ES,POINT (-6.360559178196425 39.79927682633197)
6723,way/990136230-220,220,f,Substation,f,way/990136230,-6.424683,39.507010,ES,POINT (-6.424683235104082 39.507009792321476)
6724,way/991952490-220,220,f,Substation,f,way/991952490,-6.341711,38.896533,ES,POINT (-6.341710703039476 38.896532838954116)


In [9]:
unique_tag_count = df_buses_ES['tags'].nunique()

print(f"El número de buses con diferentes valores de tags son {unique_tag_count}")

El número de buses con diferentes valores de tags son 947
